# SOLSTICE quickstart: first DIII-D state models

Train a baseline state model (control parameters -> 2D plasma fields) on the
DIII-D APP-FPP ensemble store (761 SOLPS-ITER cases, native 96x36 grid,
built with `solstice.data.store`). Evaluation follows the metrics that
matter: SSIM on the 2D fields (SPARC SOLPS-NN milestone style), and
divertor **target profiles** + peak-target error (the REACT go/no-go
quantity), plus the outer-midplane profile.

Expects the store in Drive at `MyDrive/SOLPS_DATA/`. Runs on a free Colab GPU.


In [ ]:
!pip -q install xarray netcdf4
from google.colab import drive
drive.mount('/content/drive')
STORE = '/content/drive/MyDrive/SOLPS_DATA/solstice_store_diiid_appfpp_v0.nc'
WEIGHTS_DEST = '/content/drive/MyDrive/SOLPS_DATA/solstice_weights/v2'

# ---- training knobs: defaults are a smoke run; for a real model use ----
# EPOCHS = 5000, HIDDEN = 1024, and consider a small grid over LR/HIDDEN
EPOCHS = 800
HIDDEN = 512
LR     = 1e-3


In [ ]:
import numpy as np, xarray as xr, torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

ds = xr.open_dataset(STORE)
print(dict(ds.sizes))
INPUTS = sorted(v for v in ds.data_vars if v.startswith('input_'))
print('inputs:', INPUTS)
NX, NY = ds.attrs['nx'], ds.attrs['ny']


## Helpers: mesh plot, image reconstruction, target/OMP cell selections


In [ ]:
def plot_field(values, title='', ax=None, cmap='viridis', log=False, sym=False):
    verts = np.stack([ds.cell_corners_r.values, ds.cell_corners_z.values], axis=-1)
    v = np.log10(np.clip(values, 1e-30, None)) if log else values
    ax = ax or plt.subplots(figsize=(4, 6))[1]
    kw = {}
    if sym:
        m = np.max(np.abs(v)); kw = dict(clim=(-m, m))
    pc = PolyCollection(verts, array=v, cmap=cmap, edgecolor='none', **kw)
    ax.add_collection(pc); ax.autoscale(); ax.set_aspect('equal')
    ax.set_xlabel('R (m)'); ax.set_ylabel('Z (m)'); ax.set_title(title)
    plt.colorbar(pc, ax=ax, shrink=0.8)
    return ax

def to_image(values):
    '''(cell,) -> (nx, ny) image via the structured labels (for SSIM).'''
    img = np.full((NX, NY), np.nan)
    img[ds.cell_ix.values - 1, ds.cell_iy.values - 1] = values
    return img

import json as _json
FACE_SETS = _json.loads(ds.attrs['face_sets'])
def target_cells(which):
    '''Cells adjacent to a target face set, ordered along the target.'''
    faces = ds.face_set.values == FACE_SETS[which]
    cells = ds.face_cells.values[faces, 0]
    return cells[np.argsort(ds.cell_iy.values[cells])]

INNER, OUTER = target_cells('inner_target'), target_cells('outer_target')
# outer midplane column: LFS cells nearest Z=0
lfs = ds.cell_r.values > ds.cell_r.values.mean()
ix_omp = ds.cell_ix.values[lfs][np.argmin(np.abs(ds.cell_z.values[lfs]))]
OMP = np.where(ds.cell_ix.values == ix_omp)[0]
OMP = OMP[np.argsort(ds.cell_iy.values[OMP])]
print('inner/outer target cells:', len(INNER), len(OUTER), '| OMP column ix =', int(ix_omp))


## Look at one case


In [ ]:
k = 0
fig, axs = plt.subplots(1, 2, figsize=(9, 6))
plot_field(ds.te.isel(case=k).values, 'Te (eV)', axs[0])
plot_field(ds.ne.isel(case=k).values, 'log10 ne (m^-3)', axs[1], log=True)
plt.tight_layout()


## Tensors and normalization

Inputs log-scaled where they span decades, then standardized. Fields use
log10 for densities and **per-cell** standardization (location-dependent
normalization -- target regions span orders of magnitude).


In [ ]:
FIELDS = {'te': True, 'ti': True, 'ne': True, 'ua_D1': False}  # name -> log10?
LOG_INPUTS = ('input_n_core', 'input_puff_D2', 'input_puff_Ne', 'input_dna')

X = np.stack([ds[v].values for v in INPUTS], axis=1).astype(np.float64)
for j, v in enumerate(INPUTS):
    if v in LOG_INPUTS:
        X[:, j] = np.log10(np.clip(X[:, j], 1e-30, None))
x_mean, x_std = X.mean(0), X.std(0) + 1e-12
Xn = (X - x_mean) / x_std

def prep_field(name, log):
    y = ds[name].values.astype(np.float64)
    if log:
        y = np.log10(np.clip(np.abs(y), 1e-6, None))
    mean, std = y.mean(0), y.std(0) + 1e-12   # per-cell stats
    return (y - mean) / std, mean, std

# QC columns (qc_pass, puff_record_missing) are advisory: all cases pass today.
rng = np.random.default_rng(0)
idx = rng.permutation(ds.sizes['case'])
split = int(0.85 * len(idx))
itr, ite = idx[:split], idx[split:]
print(len(itr), 'train /', len(ite), 'test cases')


## Train one MLP per field (params -> full 2D field)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_field(name, log, hidden=HIDDEN, epochs=EPOCHS, lr=LR):
    Yn, mean, std = prep_field(name, log)
    xt = torch.tensor(Xn[itr], dtype=torch.float32, device=device)
    yt = torch.tensor(Yn[itr], dtype=torch.float32, device=device)
    xv = torch.tensor(Xn[ite], dtype=torch.float32, device=device)
    yv = torch.tensor(Yn[ite], dtype=torch.float32, device=device)
    model = nn.Sequential(
        nn.Linear(len(INPUTS), hidden), nn.GELU(),
        nn.Linear(hidden, hidden), nn.GELU(),
        nn.Linear(hidden, ds.sizes['cell'])).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state = np.inf, None
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = nn.functional.mse_loss(model(xt), yt)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val = nn.functional.mse_loss(model(xv), yv).item()
        if val < best:
            best, best_state = val, {k: v.clone() for k, v in model.state_dict().items()}
        if ep % 200 == 0:
            print(f'{name} ep{ep}: train {loss.item():.4f} val {val:.4f}')
    model.load_state_dict(best_state)
    return model, mean, std, best

models = {}
for name, log in FIELDS.items():
    models[name] = train_field(name, log)
    print(name, 'best val MSE (normalized):', round(models[name][3], 5))


## Metrics over the whole test set

- **SSIM** on the reconstructed (ix, iy) images (log space for densities);
  SPARC SOLPS-NN milestone style, target > 0.95
- **R^2** and **RMSE** in the (log-)physical space the model is trained in
- **peak target error**: median |error| of the peak value at each divertor
  target (REACT-style; target < 5%)


In [ ]:
from skimage.metrics import structural_similarity as ssim

def predict_batch(name, case_idx):
    model, mean, std, _ = models[name]
    with torch.no_grad():
        yn = model(torch.tensor(Xn[case_idx], dtype=torch.float32, device=device)).cpu().numpy()
    return yn * std + mean   # (log-)physical space

def truth_batch(name, case_idx):
    y = ds[name].values[case_idx]
    return np.log10(np.clip(np.abs(y), 1e-6, None)) if FIELDS[name] else y

print(f"{'field':8s} {'SSIM':>6s} {'R2':>7s} {'RMSE':>9s} {'peak-tgt %':>11s}")
for name in FIELDS:
    P, T = predict_batch(name, ite), truth_batch(name, ite)
    ss = np.mean([ssim(to_image(t), to_image(p),
                       data_range=np.nanmax(t) - np.nanmin(t))
                  for t, p in zip(T, P)])
    r2 = 1 - np.sum((P - T)**2) / np.sum((T - T.mean())**2)
    rmse = np.sqrt(np.mean((P - T)**2))
    peak = np.median([abs(p[c].max() - t[c].max()) / abs(t[c].max()) * 100
                      for t, p in zip(T, P) for c in (INNER, OUTER)])
    unit = 'dex' if FIELDS[name] else 'phys'
    print(f'{name:8s} {ss:6.3f} {r2:7.3f} {rmse:9.3f} ({unit}) {peak:9.2f}')


## 2D comparison (absolute error, not relative)

Relative error explodes where Te ~ 0.1 eV and is misleading there;
the map below shows absolute error with a symmetric scale.


In [ ]:
k = int(ite[0])
truth = ds.te.isel(case=k).values
pred_log = predict_batch('te', [k])[0]
pred = pred_log  # te trained in linear space
fig, axs = plt.subplots(1, 3, figsize=(13, 6))
plot_field(truth, 'Te SOLPS (eV)', axs[0])
plot_field(pred, 'Te SOLSTICE-MLP (eV)', axs[1])
plot_field(pred - truth, 'error (eV)', axs[2], cmap='RdBu_r', sym=True)
plt.tight_layout()


## Divertor target profiles (the quantity that matters)


In [ ]:
def profile_plot(name, cells, label, ax):
    t = ds[name].isel(case=k).values[cells]
    p = predict_batch(name, [k])[0][cells]
    if FIELDS[name]:
        p = 10**p
    r = ds.cell_r.values[cells]
    ax.plot(r, t, 'o-', label='SOLPS')
    ax.plot(r, p, 's--', label='MLP')
    ax.set_xlabel('R (m)'); ax.set_title(label); ax.legend()

fig, axs = plt.subplots(2, 2, figsize=(11, 7))
profile_plot('te', INNER, 'inner target Te (eV)', axs[0, 0])
profile_plot('te', OUTER, 'outer target Te (eV)', axs[0, 1])
profile_plot('ne', INNER, 'inner target ne (m^-3)', axs[1, 0]); axs[1, 0].set_yscale('log')
profile_plot('ne', OUTER, 'outer target ne (m^-3)', axs[1, 1]); axs[1, 1].set_yscale('log')
for ax in axs[0]: ax.set_yscale('log')
plt.suptitle(f'case {ds.case.values[k]}'); plt.tight_layout()


## Outer-midplane profile


In [ ]:
plt.figure(figsize=(6, 4))
r = ds.cell_r.values[OMP]
plt.plot(r, ds.te.isel(case=k).values[OMP], 'o-', label='SOLPS')
plt.plot(r, predict_batch('te', [k])[0][OMP], 's--', label='MLP')
plt.xlabel('R (m)'); plt.ylabel('Te (eV)'); plt.yscale('log')
plt.title(f'OMP profile, case {ds.case.values[k]}'); plt.legend()


## Save weights to Drive

Weights + normalization stats are the ingredients of a SOLSTICE checkpoint
bundle (`docs/specs/checkpoint_spec.md`); packaging happens repo-side.


In [ ]:
import os
os.makedirs(WEIGHTS_DEST, exist_ok=True)
for name, (model, mean, std, _) in models.items():
    torch.save({'state_dict': model.state_dict(),
                'cell_mean': mean, 'cell_std': std, 'log10': FIELDS[name],
                'inputs': INPUTS, 'x_mean': x_mean, 'x_std': x_std,
                'hidden': HIDDEN, 'epochs': EPOCHS, 'train_idx_seed': 0},
               f'{WEIGHTS_DEST}/diiid-appfpp-state-mlp-{name}.pt')
print(sorted(os.listdir(WEIGHTS_DEST)))
